# Phase 2c: evaluate the fine-tuned whisper-small on dev

Runs the fine-tuned model (`best/` from notebook 03) through exactly the same evaluation as the zero-shot
baseline: `cyasr.run_baseline`, the 280 full-length dev clips, long-form sequential decoding (greedy and
fallback) and the same scoring. Only the model differs, so the numbers are directly comparable.

Test is not touched here; it is evaluated once, at the end of the project.

Before starting, run `bash scripts/make_colab_bundle.sh` locally and upload the new `cyasr_bundle.zip` to
`MyDrive/cyasr/`, replacing the old one (it must contain `configs/eval_whisper-small_lora.yaml`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BUNDLE = '/content/drive/MyDrive/cyasr/cyasr_bundle.zip'
DRIVE_RUNS = '/content/drive/MyDrive/cyasr/runs'
CONFIG = 'configs/eval_whisper-small_lora.yaml'

!rm -rf /content/cyasr && mkdir -p /content/cyasr
!unzip -q {BUNDLE} -d /content/cyasr
%cd /content/cyasr
!ls {CONFIG}  # fails if the bundle is the old one
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q -r requirements.txt
# Colab ships an old torchao that peft rejects ("only versions above 0.16.0"); we don't use it
!pip uninstall -y -q torchao

In [ ]:
# The fine-tuned model saved by notebook 03 (LoRA already merged, so it is a plain Whisper model)
import yaml
MODEL = yaml.safe_load(open(CONFIG))['model']
!ls -lh {MODEL}

In [ ]:
# Smoke test: 4 dev clips, greedy, saved locally. Checks that the model loads and writes Cypriot text.
import json
!python -m cyasr.run_baseline --config {CONFIG} --decoding greedy --max-samples 4
for line in open('runs/whisper-small_lora_greedy_dev_n4/predictions.jsonl', encoding='utf-8'):
    r = json.loads(line)
    print(f"--- {r['audio_id']} ({r['duration_s']:.0f} s)\nREF: {r['ref'][:200]}\nHYP: {r['hyp'][:200]}")

In [ ]:
# The real evaluation: all dev clips, greedy then fallback, written straight to Drive.
# Roughly 5 min (greedy) + 10 min (fallback) on an L4.
cfg = yaml.safe_load(open(CONFIG))
cfg['output_dir'] = DRIVE_RUNS
yaml.safe_dump(cfg, open('/content/eval.yaml', 'w'), allow_unicode=True)
for decoding in ['greedy', 'fallback']:
    !python -m cyasr.run_baseline --config /content/eval.yaml --decoding {decoding}

In [ ]:
# Baseline (zero-shot) vs fine-tuned, same dev clips, same decoding, same scoring
def load(name):
    return json.load(open(f'{DRIVE_RUNS}/{name}/metrics.json'))

print(f"{'decoding':9} {'model':11} {'WER basic':>9} {'WER nodiac':>10} {'CER nodiac':>10} {'halluc.':>7} {'empty':>5}")
for decoding in ['greedy', 'fallback']:
    for label, name in [('zero-shot', f'baseline_whisper-small_{decoding}_dev'),
                        ('fine-tuned', f'whisper-small_lora_{decoding}_dev')]:
        m = load(name); lv = m['levels']
        print(f"{decoding:9} {label:11} {lv['basic']['wer']:9.1%} {lv['nodiac']['wer']:10.1%} "
              f"{lv['nodiac']['cer']:10.1%} {m['hallucinations']:7} {m['empty_hypotheses']:5}")

In [ ]:
# Five clips side by side (fallback decoding): reference, zero-shot output, fine-tuned output
import random
def preds(name):
    return {r['audio_id']: r for r in map(json.loads, open(f'{DRIVE_RUNS}/{name}/predictions.jsonl', encoding='utf-8'))}

base = preds('baseline_whisper-small_fallback_dev')
ft = preds('whisper-small_lora_fallback_dev')
short = sorted(a for a in ft if ft[a]['duration_s'] <= 20)  # short clips are easier to read
for a in random.Random(0).sample(short, 5):
    print(f"--- {a} ({ft[a]['duration_s']:.0f} s)")
    print('REF       :', ft[a]['ref'])
    print('ZERO-SHOT :', base[a]['hyp'])
    print('FINE-TUNED:', ft[a]['hyp'])

## Diagnosis: does skipping silent windows remove the invented tails?

The error analysis showed that clips of 30-33 s are much worse than the rest: the last window holds well
under a second of near-silence and the model invents text for it. Here both models are run again on dev
with fallback decoding plus `no_speech_threshold 0.6` (OpenAI's default): a window is dropped when it sounds
empty AND the model is unsure of its text. If the explanation is right, the 30-33 s clips improve a lot and
the other clips stay about the same. Dev only; about 15 min on an L4.

In [ ]:
import yaml
for cfg_path in ['configs/baseline_whisper-small.yaml', 'configs/eval_whisper-small_lora.yaml']:
    cfg = yaml.safe_load(open(cfg_path))
    cfg.update(splits=['dev'], output_dir=DRIVE_RUNS)  # the baseline config also lists test; keep it untouched
    yaml.safe_dump(cfg, open('/content/diag.yaml', 'w'), allow_unicode=True)
    !python -m cyasr.run_baseline --config /content/diag.yaml --decoding fallback --no-speech-threshold 0.6

In [ ]:
import yaml
cfg = yaml.safe_load(open('configs/baseline_whisper-small.yaml'))
cfg.update(splits=['dev'], output_dir=DRIVE_RUNS)
yaml.safe_dump(cfg, open('/content/diag.yaml', 'w'), allow_unicode=True)
!python -m cyasr.run_baseline --config /content/diag.yaml --decoding fallback --no-speech-threshold 0.6

In [ ]:
# WER (nodiac) overall and by clip length, with and without the rule
import json, jiwer
from cyasr.normalize import normalize

def bucket(d):
    return '<=30s' if d <= 30 else ('30-33s' if d <= 33 else '>33s')

runs = [('zero-shot', 'off', 'baseline_whisper-small_fallback_dev'),
        ('zero-shot', '0.6', 'baseline_whisper-small_fallback_ns0.6_dev'),
        ('fine-tuned', 'off', 'whisper-small_lora_fallback_dev'),
        ('fine-tuned', '0.6', 'whisper-small_lora_fallback_ns0.6_dev')]
print(f"{'model':11} {'rule':4} {'all':>6} {'<=30s':>6} {'30-33s':>7} {'>33s':>6} {'insert.':>7} {'halluc.':>7}")
for model, rule, name in runs:
    recs = [json.loads(l) for l in open(f'{DRIVE_RUNS}/{name}/predictions.jsonl', encoding='utf-8')]
    errs, words, ins, halluc = {}, {}, 0, 0
    for r in recs:
        ref, hyp = normalize(r['ref'], 'nodiac'), normalize(r['hyp'], 'nodiac')
        o = jiwer.process_words(ref, hyp)
        for b in (bucket(r['duration_s']), 'all'):
            errs[b] = errs.get(b, 0) + o.substitutions + o.deletions + o.insertions
            words[b] = words.get(b, 0) + len(ref.split())
        ins += o.insertions
        halluc += len(hyp.split()) > 2 * max(len(ref.split()), 1)
    w = {b: errs[b] / words[b] for b in errs}
    print(f"{model:11} {rule:4} {w['all']:6.1%} {w['<=30s']:6.1%} {w['30-33s']:7.1%} {w['>33s']:6.1%} {ins:7} {halluc:7}")